In [ ]:
import csv
import json
import os
import random

# Dossier des fichiers CSV
DOSSIER_CSV = "Adjudication"
DOSSIER_SORTIE = "JSON_TrainTest"
os.makedirs(DOSSIER_SORTIE, exist_ok=True)

TAG_ID = {"PER": 1, "LOC": 2, "ORG": 3, "MISC": 4}

# Récupérer tous les fichiers CSV
fichiers = os.listdir(DOSSIER_CSV)
consolides = [f for f in fichiers if f.endswith("_consolide.csv")]
originaux = [f for f in fichiers if not f.endswith("_consolide.csv") and f.endswith(".csv")]

# Associer les consolidés à leur version originale
fichiers_associes = {}
for fichier in consolides:
    nom_base = fichier.replace("_consolide.csv", "")
    for original in originaux:
        if original.startswith(nom_base):
            fichiers_associes[nom_base] = (fichier, original)
            break

# Fonction pour traiter les fichiers
def traiter_split(fichiers_associes, train_pct):
    train_data = []
    test_data = []
    id_global = 1

    for nom_base, (fichier_consolide, fichier_original) in fichiers_associes.items():
        chemin_consolide = os.path.join(DOSSIER_CSV, fichier_consolide)
        chemin_original = os.path.join(DOSSIER_CSV, fichier_original)

        with open(chemin_consolide, newline='', encoding='utf-8') as f1, \
             open(chemin_original, newline='', encoding='utf-8') as f2:

            lecteur_consolide = list(csv.DictReader(f1, delimiter=';', quotechar='"'))
            lecteur_original = list(csv.DictReader(f2, delimiter=';', quotechar='"'))

            # Mélanger ensemble pour que ce soit aléatoire
            indices = list(range(len(lecteur_consolide)))
            random.shuffle(indices)

            taille_train = int(len(indices) * train_pct)
            indices_train = indices[:taille_train]
            indices_test = indices[taille_train:]

            for subset, indices_subset in [("train", indices_train), ("test", indices_test)]:
                tokens = []
                annotations = []
                position = 0

                for idx in indices_subset:
                    ligne_cons = lecteur_consolide[idx]
                    ligne_orig = lecteur_original[idx]

                    token = ligne_cons['Token']
                    label_maj = ligne_cons['Label_Maj'].strip()
                    model_label = ligne_orig['Label'].strip()
                    human_annotations = []

                    for cle, valeur in ligne_orig.items():
                        if cle.endswith("_Correction") and valeur.strip() in TAG_ID:
                            human_annotations.extend([v.strip() for v in valeur.split(";") if v.strip() in TAG_ID])

                    start_offset = position
                    end_offset = position + len(token)

                    if label_maj:
                        annotations.append({
                            "id": id_global,
                            "start_offset": start_offset,
                            "end_offset": end_offset,
                            "tag": {
                                "id": TAG_ID.get(label_maj, 99),
                                "name": label_maj
                            },
                            "tag_name": label_maj,
                            "model_annotations": [model_label] if model_label in TAG_ID else [],
                            "human_annotations": human_annotations
                        })
                        id_global += 1

                    tokens.append(token)
                    position = end_offset + 1

                texte = " ".join(tokens)
                document = {
                    "id": f"{nom_base}_{subset}",
                    "content": texte,
                    "metadata": {},
                    "annotations": annotations
                }

                if subset == "train":
                    train_data.append(document)
                else:
                    test_data.append(document)

        print(f"✅ {nom_base} traité pour {int(train_pct*100)}-{100-int(train_pct*100)} split")

    return train_data, test_data

# ➤ Créer les splits 80/20
print("\n📁 Création du split 80/20")
train_80, test_20 = traiter_split(fichiers_associes, train_pct=0.8)

with open(os.path.join(DOSSIER_SORTIE, "Corpus_80p.json"), "w", encoding="utf-8") as f_json:
    json.dump(train_80, f_json, indent=2, ensure_ascii=False)

with open(os.path.join(DOSSIER_SORTIE, "Corpus_20p.json"), "w", encoding="utf-8") as f_json:
    json.dump(test_20, f_json, indent=2, ensure_ascii=False)

print(f"✅ Fichiers Corpus_80p.json et Corpus_20p.json créés ({len(train_80)} + {len(test_20)} documents)")

# ➤ Créer les splits 70/30
print("\n📁 Création du split 70/30")
train_70, test_30 = traiter_split(fichiers_associes, train_pct=0.7)

with open(os.path.join(DOSSIER_SORTIE, "Corpus_70p.json"), "w", encoding="utf-8") as f_json:
    json.dump(train_70, f_json, indent=2, ensure_ascii=False)

with open(os.path.join(DOSSIER_SORTIE, "Corpus_30p.json"), "w", encoding="utf-8") as f_json:
    json.dump(test_30, f_json, indent=2, ensure_ascii=False)

print(f"✅ Fichiers Corpus_70p.json et Corpus_30p.json créés ({len(train_70)} + {len(test_30)} documents)")